In [1]:
import sys, os, math, time, csv
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    !nvidia-smi  


sys.path.insert(0, ".")
import shooter  # registers Shooter-v0

print("Environment registered: Shooter-v0")

Using device: cuda
Sat May  9 05:32:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off |   00000000:84:00.0 Off |                  N/A |
| 30%   47C    P8             29W /  250W |       4MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

In [2]:
def wrap_angle(angle: float) -> float:
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

In [10]:
import numpy as np
import gymnasium as gym
from gymnasium import Wrapper

class ShooterRewardWrapperV9_Failure(Wrapper):
    def __init__(self, env, failure_states, penalty_threshold=0.3, penalty_value=-1.0):
        super().__init__(env)
        self.failure_states = np.array(failure_states)  # shape (N, 169)
        self.penalty_threshold = penalty_threshold
        self.penalty_value = penalty_value
        self.prev_score = 0
        self.total_kills = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_score = info["hunterScore"]
        self.total_kills = 0
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

       
        shaped = reward
        score_delta = info["hunterScore"] - self.prev_score
        self.prev_score = info["hunterScore"]

        if score_delta > 0:
            shaped += score_delta * 5.0          
            self.total_kills += 1

        
        aim_err = self._calc_aim_error(obs)
        if aim_err is not None:
            shaped += 2.0 * np.exp(-aim_err**2 / 0.1)

            if aim_err < 0.08:
                shaped += 2.0                     

            if action == 1 and aim_err < 0.15:
                shaped += 1.5                     

        # ── Failure Penalty ──
        
        if len(self.failure_states) > 0:
            dists = np.linalg.norm(self.failure_states - obs, axis=1)
            min_dist = np.min(dists)
            if min_dist < self.penalty_threshold:
                shaped += self.penalty_value   

        info["kills"] = self.total_kills
        return obs, shaped, terminated, truncated, info

    def _calc_aim_error(self, obs):
       
        best_dist = 1e9
        best_x = best_y = best_z = None
        for i in range(15):
            j = 6 + i*8
            if obs[j+6] > 0.5:
                x = obs[j] * 100.0
                z = obs[j+1] * 100.0
                y = obs[j+5] * 20.0
                d = np.sqrt(x**2 + z**2)
                if d < best_dist:
                    best_dist = d
                    best_x, best_y, best_z = x, y, z
        if best_x is None:
            return None
        cur_yaw = obs[3] * np.pi
        cur_pitch = obs[4] * 0.9
        ideal_yaw = np.arctan2(best_z, best_x)
        xz = max(np.sqrt(best_x**2 + best_z**2), 0.5)
        ideal_pitch = np.arctan2(best_y - 2.3, xz)
        yaw_err = abs(wrap_angle(ideal_yaw - cur_yaw))
        pitch_err = abs(ideal_pitch - cur_pitch)
        return yaw_err + pitch_err

In [11]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback
import time, os

# failure states
failure_states = np.load("failure_states.npy")

# wrapper environment
def make_failure_env():
    env = gym.make("Shooter-v0", render_mode=None)
    return ShooterRewardWrapperV9_Failure(env, failure_states, penalty_threshold=0.3, penalty_value=-1.0)

vec_env = make_vec_env(make_failure_env, n_envs=4)


model = PPO.load("ppo_v9_failure_finetuned", env=vec_env, device="cuda") 


model.learning_rate = 1e-5


from torch.utils.tensorboard import SummaryWriter
import time

class DetailedLogCallbackWithTB(BaseCallback):
    def __init__(self, writer, log_every=100):
        super().__init__()
        self.writer = writer
        self.log_every = log_every
        self.ep_count = 0
        self.cur_rewards = {}
        self.ep_rewards = []
        self.ep_kills = []
        self.ep_scores = []
        self.start_time = time.time()

    def _on_step(self):
        infos = self.locals.get("infos", [{}] * len(self.locals["dones"]))
        for i, done in enumerate(self.locals["dones"]):
            self.cur_rewards[i] = self.cur_rewards.get(i, 0) + self.locals["rewards"][i]
            if done:
                self.ep_count += 1
                self.ep_rewards.append(self.cur_rewards[i])
                self.ep_kills.append(infos[i].get("kills", 0))
                self.ep_scores.append(infos[i].get("hunterScore", 0))
                self.cur_rewards[i] = 0

                if self.ep_count % self.log_every == 0:
                    recent_r = self.ep_rewards[-self.log_every:]
                    recent_k = self.ep_kills[-self.log_every:]
                    recent_s = self.ep_scores[-self.log_every:]
                    elapsed = time.time() - self.start_time
                    mean_r = np.mean(recent_r)
                    max_r = np.max(recent_r)
                    min_r = np.min(recent_r)
                    sum_k = int(np.sum(recent_k))
                    mean_s = np.mean(recent_s)

                    
                    print(f"Ep {self.ep_count:5d} | "
                          f"Mean: {mean_r:7.2f} | Max: {max_r:7.2f} | Min: {min_r:7.2f} | "
                          f"Kill: {sum_k:4d} | Score: {mean_s:6.1f} | "
                          f"Step {self.num_timesteps:7d} | Time: {elapsed/60:6.1f}m")

                   
                    if self.writer:
                        self.writer.add_scalar("Episode/Mean_Reward", mean_r, self.ep_count)
                        self.writer.add_scalar("Episode/Mean_Score", mean_s, self.ep_count)
                        self.writer.add_scalar("Episode/Sum_Kills", sum_k, self.ep_count)
                        self.writer.add_scalar("Training/Timesteps", self.num_timesteps, self.ep_count)
        return True

In [12]:

writer = SummaryWriter(log_dir="runs/ppo_v9_failure")


callback = DetailedLogCallbackWithTB(writer=writer, log_every=100)


model.learn(total_timesteps=1_000_000, callback=callback, progress_bar=True)


writer.close()


model.save("ppo_v9_failure_finetuned")
print("Saved ppo_v9_failure_finetuned")

Output()

Ep   100 | Mean: 4047.23 | Max: 15769.16 | Min:   76.39 | Kill: 2163 | Score:  210.3 | Step  122008 | Time:    3.4m

Ep   200 | Mean: 4935.10 | Max: 14610.41 | Min:   32.37 | Kill: 2645 | Score:  264.5 | Step  258036 | Time:    7.5m

Ep   300 | Mean: 4423.27 | Max: 14537.87 | Min:  154.70 | Kill: 2359 | Score:  208.8 | Step  385896 | Time:   11.2m

Ep   400 | Mean: 4431.88 | Max: 14778.17 | Min:   77.10 | Kill: 2358 | Score:  204.6 | Step  510480 | Time:   14.8m

Ep   500 | Mean: 4639.06 | Max: 13992.11 | Min:  143.06 | Kill: 2447 | Score:  221.6 | Step  641300 | Time:   18.6m

Ep   600 | Mean: 5839.60 | Max: 15071.05 | Min:  162.38 | Kill: 3081 | Score:  318.9 | Step  805668 | Time:   23.4m

Ep   700 | Mean: 5280.52 | Max: 15350.61 | Min:   96.47 | Kill: 2763 | Score:  275.1 | Step  944512 | Time:   27.5m

Saved ppo_v9_failure_finetuned


In [8]:
def test_30(model_path):
    model = PPO.load(model_path)
    env = gym.make("Shooter-v0", render_mode=None)
    scores = []
    for ep in range(30):
        obs, _ = env.reset()
        done = False
        ep_score = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                break
        scores.append(info["hunterScore"])
        print(f"Ep {ep+1}: Score {info['hunterScore']}")
    print(f"Mean Score: {np.mean(scores):.1f}")
    env.close()

test_30("ppo_v9_failure_finetuned")

/home/jupyter-st125843/.local/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Ep 1: Score -162
Ep 2: Score 30
Ep 3: Score 478
Ep 4: Score -36
Ep 5: Score 20
Ep 6: Score 298
Ep 7: Score 382
Ep 8: Score 498
Ep 9: Score -104
Ep 10: Score 222
Ep 11: Score -140
Ep 12: Score 312
Ep 13: Score 34
Ep 14: Score 558
Ep 15: Score 354
Ep 16: Score 4
Ep 17: Score 8
Ep 18: Score 404
Ep 19: Score -6
Ep 20: Score -18
Ep 21: Score 38
Ep 22: Score -48
Ep 23: Score 276
Ep 24: Score 8
Ep 25: Score 502
Ep 26: Score 564
Ep 27: Score -62
Ep 28: Score -108
Ep 29: Score 8
Ep 30: Score 38
Mean Score: 145.1
